# Build a Signals-powered AI agent with AWS Bedrock AgentCore

This notebook accompanies the [Signals agentic accelerator](https://docs.snowplow.io/tutorials/signals-agentic-accelerator/) tutorial. Follow the tutorial for full context and step-by-step instructions.

## Install dependencies

Install the required Python packages. This requires Python 3.11 or higher.

In [ ]:
# install dependencies, this requires Python 3.11 or higher
!pip3 install strands-agents strands-agents-tools snowplow-signals snowplow-tracker bedrock-agentcore boto3 duckduckgo-search

## Configure credentials

### Google Colab (recommended)

Store all credentials using Colab's built-in secrets manager (click the key icon in the left sidebar). The cell below loads them automatically.

**AWS credentials:**
* `AWS_ACCESS_KEY_ID` - your IAM access key ID
* `AWS_SECRET_ACCESS_KEY` - your IAM secret access key

Generate these in the [IAM console](https://console.aws.amazon.com/iam/) under **Users > your user > Security credentials > Create access key**.

**Signals Sandbox credentials:**
* `SIGNALS_API_URL` - the URL for your Signals instance
* `SIGNALS_ACCESS_TOKEN` - the access token from the [Sandbox dashboard](https://try-signals.snowplow.io/dashboard)
* `SIGNALS_COLLECTOR_URL` - your collector URL

**Signals CDI credentials** (if using CDI instead of Sandbox):
* `SIGNALS_API_URL` - the URL for your Signals instance
* `SIGNALS_API_KEY` - the API key generated in console
* `SIGNALS_API_KEY_ID` - the API key ID generated in console
* `SIGNALS_ORG_ID` - your organization ID from console
* `SIGNALS_COLLECTOR_URL` - your collector URL

### Running locally

If running locally, ensure your AWS CLI is configured (`aws configure`) and fill in the Signals variables directly in the cell below.

### Additional AWS setup

* **Region**: set `AWS_REGION` to a region that supports both Bedrock and AgentCore Memory (`us-east-1` or `us-west-2` recommended)
* **Model access**: enable the Claude model in the [Bedrock console](https://console.aws.amazon.com/bedrock/) under **Model access > Enable specific models**
* **IAM permissions**: your AWS user needs `bedrock:InvokeModel`, `bedrock-agentcore:*`, and `iam:PassRole` (scoped to `bedrock-agentcore.amazonaws.com`) permissions. AgentCore Memory creation requires `iam:PassRole`.

In [ ]:
import os
from snowplow_signals import Criteria, Criterion, Signals, Attribute, AtomicProperty, Event, EntityProperty, EventProperty, domain_sessionid, SignalsSandbox

# AWS configuration
AWS_REGION = 'us-west-2'
os.environ['AWS_DEFAULT_REGION'] = AWS_REGION

# Load credentials from Colab secrets if available
try:
    from google.colab import userdata
    os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
    os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')

    API_URL = userdata.get('SIGNALS_API_URL')
    COLLECTOR_URL = userdata.get('SIGNALS_COLLECTOR_URL')

    # Try Sandbox first, fall back to CDI
    try:
        ACCESS_TOKEN = userdata.get('SIGNALS_ACCESS_TOKEN')
    except Exception:
        ACCESS_TOKEN = None

    try:
        API_KEY = userdata.get('SIGNALS_API_KEY')
        API_KEY_ID = userdata.get('SIGNALS_API_KEY_ID')
        ORG_ID = userdata.get('SIGNALS_ORG_ID')
    except Exception:
        API_KEY = None
        API_KEY_ID = None
        ORG_ID = None

    print('Loaded credentials from Colab secrets')

except (ImportError, ModuleNotFoundError):
    # Not running in Colab - set variables directly below
    print('Not running in Colab - set credentials below')

    # Snowplow Signals CDI
    API_URL = 'example.signals.snowplowanalytics.com'
    API_KEY = ''
    API_KEY_ID = ''
    ORG_ID = ''

    # or Signals Sandbox
    API_URL = 'you.signals.snowplowanalytics.com'
    ACCESS_TOKEN = ''

    COLLECTOR_URL = 'https://your-collector.svc.snplow.net'

# Connect to Signals
sandbox = ACCESS_TOKEN is not None and ACCESS_TOKEN != ''

if not sandbox:
    print('Using Snowplow Signals CDI')
    sp_signals = Signals(
        api_url=API_URL,
        api_key=API_KEY,
        api_key_id=API_KEY_ID,
        org_id=ORG_ID
    )
else:
    print('Using Signals Sandbox')
    sp_signals = SignalsSandbox(
        api_url=API_URL,
        sandbox_token=ACCESS_TOKEN
    )

## Define the agent tools

Each tool is a Python function decorated with `@tool` from Strands Agents.

In [ ]:
import json
from strands.tools import tool
from duckduckgo_search import DDGS
from snowplow_signals import SignalsSandbox

@tool
def get_destination_info(destination: str) -> str:
    """Get information about a specific destination.

    Args:
        destination: The name of the destination (e.g., 'Bangkok')

    Returns:
        Formatted information about the destination including
        descriptions, ratings, highlights, and tags
    """
    destinations = {
        "bangkok": {
            "name": "Bangkok", "country": "Thailand", "price": 150,
            "description": "Vibrant capital with street food, temples, and bustling markets",
            "rating": 4.5, "highlights": ["Street Food", "Temples", "Nightlife", "Shopping"],
            "tags": ["temples", "nightlife", "street-food", "shopping", "urban"],
        },
        "bali": {
            "name": "Bali", "country": "Indonesia", "price": 180,
            "description": "Tropical paradise with rich Hindu culture and stunning landscapes",
            "rating": 4.7, "highlights": ["Beaches", "Culture", "Wellness", "Nature"],
            "tags": ["beaches", "wellness", "culture", "nature", "temples"],
        },
        "ho chi minh city": {
            "name": "Ho Chi Minh City", "country": "Vietnam", "price": 90,
            "description": "Dynamic city blending French colonial charm with modern energy",
            "rating": 4.3, "highlights": ["History", "Food", "Architecture", "Nightlife"],
            "tags": ["history", "street-food", "nightlife", "architecture", "urban"],
        },
        "singapore": {
            "name": "Singapore", "country": "Singapore", "price": 320,
            "description": "Modern city-state with incredible food and efficient infrastructure",
            "rating": 4.8, "highlights": ["Food", "Shopping", "Architecture", "Gardens"],
            "tags": ["urban", "shopping", "food", "family-friendly", "modern"],
        },
        "chiang mai": {
            "name": "Chiang Mai", "country": "Thailand", "price": 85,
            "description": "Cultural hub in northern Thailand with temples and mountain views",
            "rating": 4.6, "highlights": ["Temples", "Nature", "Culture", "Wellness"],
            "tags": ["temples", "nature", "culture", "wellness", "mountains"],
        },
        "kuala lumpur": {
            "name": "Kuala Lumpur", "country": "Malaysia", "price": 140,
            "description": "Multicultural capital with iconic towers and diverse cuisine",
            "rating": 4.4, "highlights": ["Food", "Shopping", "Architecture", "Culture"],
            "tags": ["urban", "shopping", "food", "architecture", "multicultural"],
        },
        "siem reap": {
            "name": "Siem Reap", "country": "Cambodia", "price": 75,
            "description": "Gateway to Angkor Wat with ancient temples and Khmer culture",
            "rating": 4.5, "highlights": ["Temples", "History", "Culture", "Architecture"],
            "tags": ["temples", "history", "culture", "architecture", "ancient"],
        },
        "luang prabang": {
            "name": "Luang Prabang", "country": "Laos", "price": 110,
            "description": "Serene UNESCO town with Buddhist temples and Mekong River views",
            "rating": 4.7, "highlights": ["Temples", "Nature", "Peace", "Culture"],
            "tags": ["temples", "nature", "peaceful", "culture", "river"],
        },
    }

    selected = destinations.get(destination.lower())
    if selected:
        return json.dumps(selected, indent=2)
    else:
        available = [d["name"] for d in destinations.values()]
        return f"Destination '{destination}' not found. Available: {available}"


@tool
def get_experience_info(experience: str) -> str:
    """Get detailed information about available experiences.

    Args:
        experience: Experience name

    Returns:
        Formatted experience information
    """
    experiences = {
        "cooking_class": {"name": "Cooking Class", "duration": "3 hours", "tags": ["food", "culture"]},
        "temple_tour": {"name": "Temple Tour", "duration": "4 hours", "tags": ["culture", "history"]},
        "island_hopping": {"name": "Island Hopping", "duration": "Full day", "tags": ["beaches", "nature"]},
        "food_tour": {"name": "Street Food Tour", "duration": "3 hours", "tags": ["food", "culture"]},
        "diving": {"name": "Scuba Diving", "duration": "Full day", "tags": ["adventure", "nature"]},
    }
    return json.dumps(experiences, indent=2)


@tool
def web_search(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.

    Args:
        keywords: The search query keywords
        region: The search region (e.g., us-en, uk-en)
        max_results: Maximum number of results to return
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except Exception as e:
        return f"Search error: {str(e)}"


# Session ID for Signals lookup - in production this comes from the application context
SIGNALS_SESSION_ID = "8a4e4985-7d98-4e61-b82e-3846e2b192fe"

@tool
def get_signals() -> str:
    """Retrieve behavioral data about the current user's activity on the site.
    Returns attributes such as page view counts, segment affinities, and preferences.
    """
    try:
        if sandbox:
            signals_client = SignalsSandbox(
                api_url=API_URL,
                sandbox_token=ACCESS_TOKEN,
            )
        else:
            signals_client = Signals(
                api_url=API_URL,
                api_key=API_KEY,
                api_key_id=API_KEY_ID,
                org_id=ORG_ID,
            )
        response = signals_client.get_service_attributes(
            name="travel_service",
            attribute_key="domain_sessionid",
            identifier=SIGNALS_SESSION_ID,
        )
        return json.dumps(response)
    except Exception as e:
        return f"Failed to get Signals: {e}"

## Configure the foundation model and create the agent

Set up the Bedrock model and combine it with the tools and a system prompt.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    temperature=0.3,
    region_name=AWS_REGION,
)

SYSTEM_PROMPT = """You are a helpful Southeast Asia travel assistant for a travel
website focused on Southeast Asian destinations. You have extensive knowledge about
destinations, experiences, practical travel information, local culture, and food.

Provide helpful, accurate, and engaging responses. Keep responses concise but
informative. Use bullet points or numbered lists when appropriate.

To personalize responses, use the get_signals tool to fetch user preferences and
behavioral data. When using these attributes, restate the user's derived preferences
and explain why your recommendations are relevant to them.

You have access to the following tools:
1. get_destination_info() - For destination information
2. get_experience_info() - For experience details
3. web_search() - For current or updated information
4. get_signals() - For retrieving behavioral data about user activity

Always use the appropriate tool to get accurate, up-to-date information rather than
making assumptions."""

agent = Agent(
    model=model,
    tools=[get_destination_info, get_experience_info, web_search, get_signals],
    system_prompt=SYSTEM_PROMPT,
)

## Test the agent

Test the agent with a sample query.

In [ ]:
response = agent("Tell me about Bangkok as a destination")

## Define behavioral attributes

Define a set of behavioral attributes based on user interactions with the travel site.

In [ ]:
# now let's define some attributes using the connection from above

travel_vendor = 'com.snowplowanalytics.accelerators.travel'

# tags pertaining to different segments
cultural_explorer_tags = ["culture", "history", "heritage", "ancient", "temples", "art", "traditional"]
modern_urbanite_tags = ["urban", "nightlife", "shopping", "modern", "architecture"]
tranquil_seeker_tags = ["nature", "peaceful", "beaches", "mountains", "river", "wellness"]
family_fun_tags = ["family-friendly", "beaches", "nature", "food", "mountains", "culture"]
culinary_tourist_tags = ["food", "street food", "multicultural", "traditional", "urban", "shopping"]


page_view_count = Attribute(
    name="page_view_count",
    type="int32",
    description="A count of all page view events",
    events=[
        Event(
            vendor="com.snowplowanalytics.snowplow",
            name="page_view",
            version="1-0-0"
        )
    ],
    aggregation="counter"
)

dest_page_view_count = Attribute(
    name="destination_page_view_count",
    type="int32",
    description="A count of destination page view events",
    events=[
        Event(
            vendor="com.snowplowanalytics.snowplow",
            name="page_view",
            version="1-0-0"
        )
    ],
    criteria=Criteria(
    all=[
        Criterion.like(
            property=AtomicProperty(name="page_urlpath"),
            value="%destinations%"
        )
    ]
    ),
    aggregation="counter"
)


family_destination_count = Attribute(
    name="family_destination_count",
    type="int32",
    description="A count of all interactions with family destination tags and pages.",
    events=[
        Event(
            vendor=travel_vendor,
            name="filter_tag_applied",
            version="1-0-0"
        ),
        Event(
            vendor='com.snowplowanalytics.snowplow',
            name='page_view',
            version="1-0-0"
        )
    ],
    aggregation="counter",
    criteria=Criteria(
        any=[
            (
                Criterion.eq(
                    property=EventProperty(
                        vendor=travel_vendor,
                        name="filter_tag_applied",
                        major_version=1,
                        path="tag_name"
                    ),
                    value="Family"
                )
            ),
            (
                Criterion.in_list(
                    property=EntityProperty(
                        vendor=travel_vendor,
                        name="content",
                        major_version=1,
                        path="primary_tag"
                    ),
                    values=family_fun_tags
                )
            )
        ]
    )
)

cultural_explorer = Attribute(
    name="cultural_explorer",
    type="int32",
    events=[
        Event(
            vendor=travel_vendor,
            name="filter_tag_applied",
            version="1-0-0",
        ),
        Event(
            vendor="com.snowplowanalytics.snowplow",
            name="page_view",
            version="1-0-0"
        )
    ],
    aggregation="counter",
    criteria=Criteria(
        any=[
            (
                Criterion.in_list(
                    property=EventProperty(
                        vendor=travel_vendor,
                        name="filter_tag_applied",
                        major_version=1,
                        path="tag_name"
                    ),
                    values=cultural_explorer_tags
                )
            ),
            (
                Criterion.in_list(
                    property=EntityProperty(
                        vendor=travel_vendor,
                        name="content",
                        major_version=1,
                        path="primary_tag"
                    ),
                    values=cultural_explorer_tags
                )
            )
        ]
    )
)

modern_urbanite = Attribute(
    name="modern_urbanite",
    type="int32",
    events=[
        Event(
            vendor=travel_vendor,
            name="filter_tag_applied",
            version="1-0-0",
        ),
        Event(
            vendor="com.snowplowanalytics.snowplow",
            name="page_view",
            version="1-0-0"
        )
    ],
    aggregation="counter",
    criteria=Criteria(
        any=[
            (
                Criterion.in_list(
                    property=EventProperty(
                        vendor=travel_vendor,
                        name="filter_tag_applied",
                        major_version=1,
                        path="tag_name"
                    ),
                    values=modern_urbanite_tags
                )
            ),
            (
                Criterion.in_list(
                    property=EntityProperty(
                        vendor=travel_vendor,
                        name="content",
                        major_version=1,
                        path="primary_tag"
                    ),
                    values=modern_urbanite_tags
                )
            )
        ]
    )
)

tranquil_seeker = Attribute(
    name="tranquil_seeker",
    type="int32",
    events=[
        Event(
            vendor=travel_vendor,
            name="filter_tag_applied",
            version="1-0-0",
        ),
        Event(
            vendor="com.snowplowanalytics.snowplow",
            name="page_view",
            version="1-0-0",
        )
    ],
    aggregation="counter",
    criteria=Criteria(
        any=[
            (
                Criterion.in_list(
                    property=EventProperty(
                        vendor=travel_vendor,
                        name="filter_tag_applied",
                        major_version=1,
                        path="tag_name"
                    ),
                    values=tranquil_seeker_tags
                )
            ),
            (
                Criterion.in_list(
                    property=EntityProperty(
                        vendor=travel_vendor,
                        name="content",
                        major_version=1,
                        path="primary_tag"
                    ),
                    values=tranquil_seeker_tags
                )
            )
        ]
    )
)

family_fun = Attribute(
    name="family_fun",
    type="int32",
    events=[
        Event(
            vendor=travel_vendor,
            name="filter_tag_applied",
            version="1-0-0",
        ),
        Event(
            vendor="com.snowplowanalytics.snowplow",
            name="page_view",
            version="1-0-0",
        )
    ],
    aggregation="counter",
    criteria=Criteria(
        any=[
            (
                Criterion.in_list(
                    property=EventProperty(
                        vendor=travel_vendor,
                        name="filter_tag_applied",
                        major_version=1,
                        path="tag_name"
                    ),
                    values=family_fun_tags
                )
            ),
            (
                Criterion.in_list(
                    property=EntityProperty(
                        vendor=travel_vendor,
                        name="content",
                        major_version=1,
                        path="primary_tag"
                    ),
                    values=family_fun_tags
                )
            )
        ]
    )
)


culinary_tourist = Attribute(
    name="culinary_tourist",
    type="int32",
    events=[
        Event(
            vendor=travel_vendor,
            name="filter_tag_applied",
            version="1-0-0",
        ),
        Event(
            vendor="com.snowplowanalytics.snowplow",
            name="page_view",
            version="1-0-0",
        )
    ],

    aggregation="counter",
    criteria=Criteria(
        any=[
            (
                Criterion.in_list(
                    property=EventProperty(
                        vendor=travel_vendor,
                        name="filter_tag_applied",
                        major_version=1,
                        path="tag_name"
                    ),
                    values=culinary_tourist_tags
                )
            ),
            (
                Criterion.in_list(
                    property=EntityProperty(
                        vendor=travel_vendor,
                        name="content",
                        major_version=1,
                        path="primary_tag"
                    ),
                    values=culinary_tourist_tags
                )
            )
        ]
    )
)

preferred_experience_length = Attribute(
    name="preferred_experience_length",
    type="string",
    events=[
        Event(
            vendor="com.snowplowanalytics.accelerators.travel",
            name="experience_filter",
            version="1-0-0",
        )
    ],
    property=EventProperty(
        vendor="com.snowplowanalytics.accelerators.travel",
        name="experience_filter",
        major_version=1,
        path="filter_value"
    ),
    aggregation="last",
    criteria=Criteria(
        all=[
            (
                Criterion.eq(
                    property=EventProperty(
                        vendor="com.snowplowanalytics.accelerators.travel",
                        name="experience_filter",
                        major_version=1,
                        path="filter_type"
                    ),
                    value="duration"
                )
            )
        ]
    )
)

budget_conscious = Attribute(
    name="budget_conscious_count",
    type="int32",
    events=[
        Event(
            vendor="com.snowplowanalytics.accelerators.travel",
            name="sort_change",
            version="1-0-0",
        ),
        Event(
            vendor="com.snowplowanalytics.accelerators.travel",
            name="destination_filter",
            version="1-0-0",
        ),
        Event(
            vendor="com.snowplowanalytics.snowplow",
            name="page_view",
            version="1-0-0",
        )
    ],
    aggregation="counter",
    criteria=Criteria(
        any=[
            (
                Criterion.eq(
                    property=EventProperty(
                        vendor="com.snowplowanalytics.accelerators.travel",
                        name="sort_change",
                        major_version=1,
                        path="sort_option"
                    ),
                    value="price-low"
                )
            ),
            (
                Criterion.eq(
                    property=EventProperty(
                        vendor="com.snowplowanalytics.accelerators.travel",
                        name="destination_filter",
                        major_version=1,
                        path="filter_value"
                    ),
                    value="budget"
                )
            ),
            (
                Criterion.in_list(
                    property=EntityProperty(
                        vendor=travel_vendor,
                        name="content",
                        major_version=1,
                        path="primary_tag"
                    ),
                    values=["budget", "budget-friendly", "affordable"]
                )
            )
        ]
    )
)

luxury_inclined = Attribute(
    name="luxury_inclined_count",
    type="int32",
    events=[
        Event(
            vendor="com.snowplowanalytics.accelerators.travel",
            name="sort_change",
            version="1-0-0",
        ),
        Event(
            vendor="com.snowplowanalytics.accelerators.travel",
            name="destination_filter",
            version="1-0-0",
        ),
        Event(
            vendor="com.snowplowanalytics.snowplow",
            name="page_view",
            version="1-0-0",
        )
    ],
    aggregation="counter",
    criteria=Criteria(
        any=[
            (
                Criterion.eq(
                    property=EventProperty(
                        vendor="com.snowplowanalytics.accelerators.travel",
                        name="sort_change",
                        major_version=1,
                        path="sort_option"
                    ),
                    value="price-high"
                )
            ),
            (
                Criterion.eq(
                    property=EventProperty(
                        vendor="com.snowplowanalytics.accelerators.travel",
                        name="destination_filter",
                        major_version=1,
                        path="filter_value"
                    ),
                    value="luxury"
                )
            ),
            (
                Criterion.in_list(
                    property=EntityProperty(
                        vendor=travel_vendor,
                        name="content",
                        major_version=1,
                        path="primary_tag"
                    ),
                    values=["luxury", "high-end"]
                )
            )
        ]
    )
)

latest_schedule = Attribute(
    name="latest_schedule",
    type="string",
    events=[
        Event(
            vendor="com.snowplowanalytics.accelerators.travel",
            name="schedule_update",
            version="1-0-0",
        )
    ],
    aggregation="last",
    property=EventProperty(
        vendor="com.snowplowanalytics.accelerators.travel",
        name="schedule_update",
        major_version=1,
        path="schedule"
    )
)

## Create an attribute group and service

Group the attributes and create a service to expose them via the Profiles API.

In [ ]:
from snowplow_signals import StreamAttributeGroup, Service

session_attributes_group = StreamAttributeGroup(
    name="travel_view",
    version=1,
    attribute_key=domain_sessionid,
    attributes=[page_view_count, dest_page_view_count, family_destination_count, cultural_explorer,
                modern_urbanite, tranquil_seeker, family_fun, culinary_tourist,
                preferred_experience_length, budget_conscious, luxury_inclined, latest_schedule],
    owner='you@email.com',
)

travel_service = Service(
    name="travel_service",
    description="Behavioral profile service for agent personalization.",
    attribute_groups=[session_attributes_group],
    owner='you@email.com'
)

response = sp_signals.publish([session_attributes_group, travel_service])
print(response)

## Validate attribute computation

Send synthetic test events using the Snowplow tracker to verify that the attributes compute correctly.
These events are sent directly to your collector - no demo site or web application is needed.
The `page_url` field is metadata used by the attribute criteria to match against URL patterns.

In [ ]:
from snowplow_tracker import Emitter, Subject, PageView, Tracker, SelfDescribingJson, SelfDescribing

user_id = "8a4e4985-7d98-4e61-b82e-3846e2b192fe"
session_id = user_id
travel_vendor = "com.snowplowanalytics.accelerators.travel"

subject = Subject()
subject.set_domain_user_id(user_id)
subject.set_domain_session_id(session_id)

tracker = Tracker(
    namespace="nb-demo",
    subject=subject,
    emitters=Emitter(endpoint=COLLECTOR_URL),
)

# Page views on destination pages
for _ in range(5):
    tracker.track(PageView(page_url="https://example.com/destinations"))

# Cultural content page views (with content entity containing cultural tags)
for tag in ["culture", "history", "temples"]:
    tracker.track(PageView(
        page_url="https://example.com/destinations/siem-reap",
        event_subject=subject,
        context=[
            SelfDescribingJson(
                f"iglu:{travel_vendor}/content/jsonschema/1-0-0",
                {"primary_tag": tag}
            )
        ]
    ))

# Family content page views
for tag in ["family-friendly", "beaches"]:
    tracker.track(PageView(
        page_url="https://example.com/destinations/bali",
        event_subject=subject,
        context=[
            SelfDescribingJson(
                f"iglu:{travel_vendor}/content/jsonschema/1-0-0",
                {"primary_tag": tag}
            )
        ]
    ))

# Filter tag applied events (cultural + food)
for tag in ["culture", "temples", "food", "street food"]:
    tracker.track(SelfDescribing(
        SelfDescribingJson(
            f"iglu:{travel_vendor}/filter_tag_applied/jsonschema/1-0-0",
            {"tag_name": tag}
        )
    ))

# Sort by price-low (budget conscious)
for _ in range(2):
    tracker.track(SelfDescribing(
        SelfDescribingJson(
            f"iglu:{travel_vendor}/sort_change/jsonschema/1-0-0",
            {"sort_option": "price-low"}
        )
    ))

# Experience filter (duration)
tracker.track(SelfDescribing(
    SelfDescribingJson(
        f"iglu:{travel_vendor}/experience_filter/jsonschema/1-0-0",
        {"filter_type": "duration", "filter_value": "half-day"}
    )
))

tracker.flush()
print("Sent test events: page views, cultural/family content, filter tags, sort, experience filter")

In [ ]:
response = sp_signals.get_service_attributes(
    name="travel_service",
    attribute_key="domain_sessionid",
    identifier=user_id,
)

print("Page view count:", response.get("page_view_count"))
print("All attributes:", response)

## Create AgentCore Memory resources

Create a memory resource with two strategies: USER_PREFERENCE for learning customer preferences, and SEMANTIC for storing factual conversation context.

In [ ]:
import boto3
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

memory_client = MemoryClient(region_name=AWS_REGION)
memory_name = "TravelAgentMemory"

# Check if memory already exists
control_client = boto3.client('bedrock-agentcore-control', region_name=AWS_REGION)
existing = control_client.list_memories()
memory_id = None
for mem in existing.get('memories', []):
    if mem['id'].startswith(memory_name):
        memory_id = mem['id']
        print(f'Found existing memory: {memory_id}')
        break

if not memory_id:
    strategies = [
        {
            StrategyType.USER_PREFERENCE.value: {
                "name": "CustomerPreferences",
                "description": "Captures customer preferences and behavior",
                "namespaces": ["travel/customer/{actorId}/preferences"],
            }
        },
        {
            StrategyType.SEMANTIC.value: {
                "name": "CustomerTravelSemantic",
                "description": "Stores facts from conversations",
                "namespaces": ["travel/customer/{actorId}/semantic"],
            }
        },
    ]

    response = memory_client.create_memory_and_wait(
        name=memory_name,
        description="Travel agent memory",
        strategies=strategies,
        event_expiry_days=90,
    )

    memory_id = response["id"]
    print(f"Created memory resource: {memory_id}")

## Seed customer history

Load sample previous interactions to demonstrate how AgentCore Memory transforms conversations into long-term customer insights.

In [ ]:
CUSTOMER_ID = "customer_001"

previous_interactions = [
    ("I'd love to go and visit somewhere warm.", "USER"),
    ("I can help with that! For warm destinations, I'd recommend Bali, Thailand, or the Philippines. Do you have a specific country in mind?", "ASSISTANT"),
    ("I'm looking for experiences that are a little bit more adventurous", "USER"),
    ("Great - extreme sports or something a little bit more down to earth?", "ASSISTANT"),
    ("Something a little tamer than extreme sports please! Ideally something that is fun but I can also do with the family.", "USER"),
]

memory_client.create_event(
    memory_id=memory_id,
    actor_id=CUSTOMER_ID,
    session_id="previous_session",
    messages=previous_interactions,
)
print("Seeded customer history successfully")

## Integrate memory hooks

Implement memory hooks that automatically retrieve customer context before each query and save interactions after each response.

In [ ]:
import logging
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

logger = logging.getLogger(__name__)


class TravelAgentMemoryHooks(HookProvider):
    """Memory hooks for automatic context retrieval and storage."""

    def __init__(self, memory_id, client, actor_id, session_id):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        """Retrieve relevant memories before the agent processes a query."""
        messages = event.agent.messages
        if (
            messages[-1]["role"] == "user"
            and "toolResult" not in messages[-1]["content"][0]
        ):
            user_query = messages[-1]["content"][0]["text"]
            try:
                all_context = []
                for context_type, namespace in self.namespaces.items():
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,
                    )
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(f"[{context_type.upper()}] {text}")

                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = (
                        f"Customer Context:\n{context_text}\n\n{original_text}"
                    )
                    logger.info(f"Retrieved {len(all_context)} customer context items")
            except Exception as e:
                logger.error(f"Failed to retrieve customer context: {e}")

    def save_travel_interaction(self, event: AfterInvocationEvent):
        """Save the interaction to memory after the agent responds."""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                customer_query = None
                agent_response = None
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif (
                        msg["role"] == "user"
                        and not customer_query
                        and "toolResult" not in msg["content"][0]
                    ):
                        customer_query = msg["content"][0]["text"]
                        break

                if customer_query and agent_response:
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[
                            (customer_query, "USER"),
                            (agent_response, "ASSISTANT"),
                        ],
                    )
                    logger.info("Saved travel interaction to memory")
        except Exception as e:
            logger.error(f"Failed to save travel interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register the memory hooks with the agent."""
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_travel_interaction)

## Test personalized responses

Rebuild the agent with memory hooks enabled and test personalized responses. The agent now has access to both:
- **Memory** - the seeded customer history from the previous step
- **Signals** - the behavioral attributes computed from the test events you sent earlier

To connect Signals data to this test, pass the same user ID you used when emitting test events. The system prompt instructs the agent to call `get_signals` and incorporate the attributes into its responses.

In [ ]:
import uuid

session_id = str(uuid.uuid4())

memory_hooks = TravelAgentMemoryHooks(
    memory_id=memory_id,
    client=memory_client,
    actor_id=CUSTOMER_ID,
    session_id=session_id,
)

agent_with_memory = Agent(
    model=model,
    hooks=[memory_hooks],
    tools=[get_destination_info, get_experience_info, web_search, get_signals],
    system_prompt=SYSTEM_PROMPT,
)

In [ ]:
response = agent_with_memory("Can you recommend some destinations for me?")

In [ ]:
response = agent_with_memory("Based on my browsing behavior, what experiences would suit me?")

## Clean up 

Note: Only run this after you are finished running the demo app

Delete the AgentCore Memory resource to avoid ongoing costs.

In [ ]:
memory_client.gmcp_client.delete_memory(memoryId=memory_id)
print(f"Deleted memory resource: {memory_id}")

In [ ]:
# Optional: remove the Signals service and attribute group
sp_signals.unpublish([travel_service, session_attributes_group])
sp_signals.delete([travel_service, session_attributes_group])
print("Deleted Signals service and attribute group")